In [ ]:
# ==================== CELL 1: Dependencies ====================
# This cell installs all packages and then AUTO-RESTARTS the runtime.
# After restart, run Cell 2 onwards (do NOT re-run Cell 1).

import subprocess, sys, importlib

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))

# Step 1: Uninstall conflicting Colab pre-installs
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
    "torch", "torchvision", "torchaudio",
    "transformers", "trl", "peft", "accelerate", "bitsandbytes"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("✅ Old packages removed")

# Step 2: PyTorch pinned to CUDA 12.1
pip("torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu121")
print("✅ PyTorch 2.5.1 installed")

# Step 3: Training stack — exact versions
pip("transformers==4.46.3",
    "trl==0.15.0",
    "peft==0.13.2",
    "accelerate==1.2.1",
    "bitsandbytes==0.45.0")
print("✅ Training stack installed")

# Step 4: Remaining deps
pip("datasets", "matplotlib", "scipy",
    "huggingface_hub", "pydantic", "openai")
print("✅ Utilities installed")

# Step 5: mergekit needs --no-build-isolation
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "--no-build-isolation", "mergekit"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("✅ mergekit installed")

print()
print("=" * 60)
print("✅ ALL DONE — restarting runtime now...")
print("   After restart, run from Cell 2 onwards.")
print("=" * 60)

# Auto-restart so new package versions are actually loaded into memory
import os, signal
os.kill(os.getpid(), signal.SIGKILL)


✅ Old packages removed
✅ PyTorch 2.5.1 installed
✅ Training stack installed
✅ Utilities installed


In [1]:
# ==================== CELL 2: Imports & Setup ====================
import subprocess, os, sys, json, re, random, logging, inspect, time, types
from collections import defaultdict
import numpy as np, torch

# Use /content (Colab's writable root) instead of /app/output
WORK = os.environ.get('WORK', '/content/output')
REPO = f'{WORK}/edupath'
SAVE_DIR = f'{WORK}/grpo_multitask'
FINAL    = f'{WORK}/grpo_final_v3'
for d in [SAVE_DIR, FINAL]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(REPO):
    for _attempt in range(3):
        result = subprocess.run(
            ['git', 'clone',
             'https://huggingface.co/spaces/degree-checker-01/meta-new-space', REPO],
            capture_output=True, text=True)
        if result.returncode == 0:
            break
        print(f'  git clone attempt {_attempt+1}/3 failed: {result.stderr.strip()}')
        if _attempt < 2:
            time.sleep(5)
    else:
        raise RuntimeError(f'git clone failed after 3 attempts: {result.stderr}')

os.chdir(REPO)
sys.path.insert(0, f'{REPO}/backend')

# Mock llm_blender — TRL 0.15.0 inspects __spec__ via find_spec()
from importlib.machinery import ModuleSpec
for m in ['llm_blender', 'llm_blender.agents', 'llm_blender.pair_ranker']:
    if m not in sys.modules:
        _mod = types.ModuleType(m)
        _mod.__spec__    = ModuleSpec(m, None)
        _mod.__path__    = []
        _mod.__package__ = m.split('.')[0]
        _mod.__file__    = None
        sys.modules[m]   = _mod

# Mock vllm — TRL 0.15.0 imports it unconditionally even when use_vllm=False
for m in ['vllm', 'vllm.sampling_params']:
    if m not in sys.modules:
        _mod = types.ModuleType(m)
        _mod.__spec__    = ModuleSpec(m, None)
        _mod.__path__    = []
        _mod.__package__ = m.split('.')[0]
        _mod.__file__    = None
        sys.modules[m]   = _mod

class _LLM:
    def __init__(self, *a, **kw):
        raise RuntimeError("vllm not installed — use_vllm must be False")

class _SamplingParams:
    def __init__(self, *a, **kw):
        pass

sys.modules['vllm'].LLM            = _LLM
sys.modules['vllm'].SamplingParams = _SamplingParams

from environment.env        import EduPathEnv
from environment.models     import Action, ActionType
from environment.student    import student_manager
from environment.curriculum import TOPIC_GRAPH, RESOURCE_DB

ALL_TOPICS   = set(TOPIC_GRAPH.keys())
FIELD_TOPICS = defaultdict(list)
TOPIC_PREREQS = {}
for tid, t in TOPIC_GRAPH.items():
    FIELD_TOPICS[t.field].append(tid)
    TOPIC_PREREQS[tid] = t.prerequisites

print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print(f'Topics: {len(ALL_TOPICS)} | Fields: {list(FIELD_TOPICS.keys())}')


GPU: Tesla T4 | VRAM: 15.6GB
Topics: 36 | Fields: ['tech', 'healthcare', 'business', 'law', 'design']


In [2]:
# ==================== CELL 3: Reward Functions ====================
_stats = {'parse_fail': 0, 'actions': defaultdict(int), 'total': 0}

TASK_TAG = {'[ACTION]': 'action', '[ROADMAP]': 'roadmap',
            '[QUIZ]': 'quiz', '[RESOURCE]': 'resource'}

def detect_task(prompt):
    if isinstance(prompt, list):
        for msg in prompt:
            if isinstance(msg, dict) and msg.get('role') == 'user':
                content = msg.get('content', '')
                for tag, task in TASK_TAG.items():
                    if tag in content[:200]:
                        return task
        return 'action'
    for tag, task in TASK_TAG.items():
        if tag in str(prompt)[:200]:
            return task
    return 'action'

def check_prereq_order(topic_list):
    seen, violations = set(), 0
    for tid in topic_list:
        for p in TOPIC_PREREQS.get(tid, []):
            if p not in seen and p in set(topic_list):
                violations += 1
        seen.add(tid)
    return violations

def reward_action(d, prompt):
    atype = d.get('type', '');  tid = d.get('topic_id')
    if isinstance(tid, (int, float)): tid = None
    valid = {a.value for a in ActionType}
    if atype not in valid: return -0.3
    r = 0.1
    m  = re.search(r'Completed:\s*\[([^\]]*)\]', str(prompt))
    nc = len([x for x in m.group(1).split(',') if x.strip()]) if m and m.group(1).strip() else 0
    m2 = re.search(r'Available:\s*\[([^\]]*)\]', str(prompt))
    avail = [x.strip().strip("'\"") for x in m2.group(1).split(',') if x.strip()] if m2 and m2.group(1).strip() else []
    m3 = re.search(r'Job readiness:\s*([\d.]+)', str(prompt))
    jr = float(m3.group(1)) if m3 else 0.0
    m4 = re.search(r'Current topic:\s*(\S+)', str(prompt))
    cur = m4.group(1) if m4 else 'None'
    if atype == 'recommend_topic':
        r += 0.7 if (tid and tid in avail) else (0.15 if (tid and tid in ALL_TOPICS) else -0.1)
    elif atype == 'assign_quiz':
        r += 0.5 if cur != 'None' else (0.2 if nc > 0 else -0.2)
    elif atype == 'assign_mini_project':
        r += 0.5 if nc >= 3 else (0.1 if nc >= 1 else -0.3)
    elif atype == 'assign_capstone':
        r += 0.7 if (nc >= 5 and jr >= 0.4) else (0.1 if nc >= 3 else -0.4)
    elif atype == 'recommend_resource':
        r += 0.3 if cur != 'None' else 0.05
    elif atype == 'mark_job_ready':
        r += 0.9 if jr >= 0.8 else (-0.1 if jr >= 0.5 else -0.5)
    if tid and tid in ALL_TOPICS: r += 0.1
    _stats['actions'][atype] += 1; _stats['total'] += 1
    if _stats['total'] > 50:
        freq = _stats['actions'][atype] / _stats['total']
        if freq > 0.5: r -= 0.15 * (freq - 0.5)
    return r

def reward_roadmap(d, prompt):
    r = 0.0; roadmap = d.get('roadmap', d.get('topics', []))
    if not isinstance(roadmap, list) or len(roadmap) == 0: return -0.3
    r += 0.2 if 3 <= len(roadmap) <= 12 else 0.05
    valid_ids = 0
    for item in roadmap:
        if isinstance(item, dict):
            tid = item.get('topic_id', '')
            if tid in ALL_TOPICS: valid_ids += 1
            if item.get('reason') or item.get('description'): r += 0.02
    r += min(0.3, valid_ids * 0.05)
    tids = [i.get('topic_id', '') for i in roadmap if isinstance(i, dict)]
    v = check_prereq_order(tids)
    r += 0.2 if v == 0 else max(0, 0.2 - v * 0.05)
    m = re.search(r'field[\"\s:]+(\w+)', str(prompt), re.I)
    if m:
        field = m.group(1).lower()
        r += min(0.15, sum(1 for t in tids if t in FIELD_TOPICS.get(field, [])) * 0.03)
    return r

def reward_quiz(d, prompt):
    r = 0.0; qs = d.get('questions', [])
    if not isinstance(qs, list) or len(qs) == 0: return -0.3
    r += 0.2 if 3 <= len(qs) <= 5 else (0.1 if 1 <= len(qs) <= 7 else 0)
    for q in qs:
        if not isinstance(q, dict): continue
        if q.get('question') and len(str(q['question'])) > 10: r += 0.05
        opts = q.get('options', [])
        if isinstance(opts, list) and len(opts) == 4: r += 0.05
        if q.get('correct', q.get('correct_answer', q.get('answer'))) is not None: r += 0.05
        if q.get('explanation'): r += 0.03
    return r

def reward_resource(d, prompt):
    r = 0.0; res = d.get('resources', [])
    if not isinstance(res, list) or len(res) == 0: return -0.3
    r += 0.15 if 2 <= len(res) <= 5 else 0
    GOOD_DOMAINS = ['coursera','udemy','docs.python','github','youtube',
                    'khanacademy','mit.edu','arxiv','tensorflow','pytorch',
                    'scikit-learn','kaggle','fast.ai','deeplearning.ai']
    for item in res:
        if not isinstance(item, dict): continue
        if item.get('title'): r += 0.05
        if item.get('url') and 'http' in str(item.get('url','')): r += 0.05
        if item.get('type') in ['course','article','video','tutorial','documentation','platform']: r += 0.03
        if item.get('reason') or item.get('description'): r += 0.03
        url = str(item.get('url',''))
        if any(dom in url for dom in GOOD_DOMAINS): r += 0.05
    m = re.search(r'topic[\"\s:]+(\w+)', str(prompt), re.I)
    if m:
        topic = m.group(1)
        topic_node = TOPIC_GRAPH.get(topic)
        for item in res:
            title = str(item.get('title',''))
            if topic_node and topic_node.name.lower() in title.lower(): r += 0.03
            if topic in RESOURCE_DB:
                real_urls = {res_item.url for res_item in RESOURCE_DB[topic]}
                if item.get('url') in real_urls: r += 0.08
    return r

def multitask_reward(completions, prompts=None, **kwargs):
    rewards = []
    for i, comp in enumerate(completions):
        if isinstance(comp, list):
            text = ''
            for msg in comp:
                if isinstance(msg, dict) and msg.get('role') == 'assistant':
                    text = msg.get('content', ''); break
            if not text and comp:
                text = comp[-1].get('content','') if isinstance(comp[-1], dict) else str(comp[-1])
        elif isinstance(comp, dict): text = comp.get('content', str(comp))
        else: text = str(comp)
        for tok in ['<|im_start|>assistant','<|im_end|>','<|endoftext|>']:
            text = text.replace(tok, '')
        text = text.strip()
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if not match:
            _stats['parse_fail'] += 1
            rewards.append(-0.5 + random.uniform(-0.05, 0.05)); continue
        try:
            d = json.loads(match.group())
        except Exception:
            try:
                fixed = re.sub(r',\s*}', '}', re.sub(r',\s*]', ']', match.group()))
                d = json.loads(fixed)
            except Exception:
                _stats['parse_fail'] += 1
                rewards.append(-0.4 + random.uniform(-0.05, 0.05)); continue
        prompt_text = ''
        if prompts is not None:
            p = prompts[min(i, len(prompts)-1)]
            if isinstance(p, list):
                for msg in p:
                    if isinstance(msg, dict) and msg.get('role') == 'user':
                        prompt_text = msg.get('content', ''); break
            if not prompt_text:
                prompt_text = p if isinstance(p, str) else str(p)
        task = detect_task(prompt_text)
        if task == 'action':   r = reward_action(d, prompt_text)
        elif task == 'roadmap': r = reward_roadmap(d, prompt_text)
        elif task == 'quiz':    r = reward_quiz(d, prompt_text)
        elif task == 'resource': r = reward_resource(d, prompt_text)
        else: r = 0.0
        rewards.append(max(min(r, 1.0), -1.0))
    if len(rewards) > 1 and float(np.std(rewards)) < 0.01:
        rewards = [r + random.uniform(-0.05, 0.05) for r in rewards]
    return rewards

# Quick smoke test
print('Reward tests:')
tests = [
    ('[ACTION] Completed: []\nAvailable: [python_basics]',
     '{"type":"recommend_topic","topic_id":"python_basics"}'),
    ('[ROADMAP] field: tech',
     '{"roadmap":[{"topic_id":"python_basics","reason":"foundation"}]}'),
    ('[QUIZ] topic: python_basics',
     '{"questions":[{"question":"What is Python?","options":["A","B","C","D"],"correct":0,"explanation":"..."}]}'),
    ('[RESOURCE] topic: python_basics',
     '{"resources":[{"title":"Python Tutorial","url":"https://docs.python.org","type":"course","reason":"official"}]}'),
]
for p, t in tests:
    r = multitask_reward([t], prompts=[p])[0]
    print(f'  {p[:30]:30s} → r={r:+.3f}')
_stats['actions'].clear(); _stats['total'] = 0; _stats['parse_fail'] = 0
print('Reward ✅')


Reward tests:
  [ACTION] Completed: []
Availab → r=+0.900
  [ROADMAP] field: tech          → r=+0.350
  [QUIZ] topic: python_basics    → r=+0.280
  [RESOURCE] topic: python_basic → r=+0.210
Reward ✅


In [3]:
# ==================== CELL 4: Multi-Task Dataset ====================
from datasets import Dataset

PROFILES = [
  {'field':'tech',       'goal':'ML Engineer',        'hours':10, 'skills':[]},
  {'field':'tech',       'goal':'Data Analyst',        'hours':15, 'skills':[{'skill':'Python','level':'beginner'}]},
  {'field':'tech',       'goal':'Web Developer',       'hours':8,  'skills':[]},
  {'field':'healthcare', 'goal':'AI in Medicine',      'hours':10, 'skills':[]},
  {'field':'business',   'goal':'Business Analytics',  'hours':12, 'skills':[]},
  {'field':'law',        'goal':'Legal AI',            'hours':8,  'skills':[]},
  {'field':'design',     'goal':'UX Designer',         'hours':10, 'skills':[]},
]

def make_action_prompt(obs):
    avail = obs.get('available_topics', [])[:6]
    return f"""[ACTION] You are an AI tutoring agent. Choose the BEST next action for this student.
RULES:
- recommend_topic: pick from Available. Best when student needs new material.
- assign_quiz: when student has a current_topic to test.
- assign_mini_project: when 2+ topics completed.
- assign_capstone: when job_readiness >= 0.4 and 5+ completed.
- recommend_resource: supplements current study.
- mark_job_ready: ONLY when job_readiness >= 0.8.
STATE:
- Completed: {obs.get('completed_topics',[])}
- Available: {avail}
- Job readiness: {obs.get('job_readiness_score',0):.2f}
- Current topic: {obs.get('current_topic','None')}
VALID TOPIC IDS: {avail}
Respond ONLY with JSON: {{"type":"<action>","topic_id":"<from_available>"}}"""

def make_roadmap_prompt(field, goal):
    topics = FIELD_TOPICS.get(field, FIELD_TOPICS['tech'])[:8]
    return f"""[ROADMAP] Create a personalized learning roadmap for a student.
Student field: {field}
Student goal: {goal}
Available topics in this field: {topics}
Respond with JSON: {{"roadmap": [{{"topic_id": "<real_id>", "name": "<topic_name>", "reason": "<why_this_topic>", "estimated_hours": <hours>}}, ...]}}
Order topics by prerequisites. Use ONLY real topic IDs from the list above."""

def make_quiz_prompt(topic_id):
    t = TOPIC_GRAPH.get(topic_id)
    name = t.name if t else topic_id
    return f"""[QUIZ] Generate a quiz for the topic: {name} (id: {topic_id})
Create 4 multiple-choice questions testing understanding of {name}.
Respond with JSON: {{"questions": [{{"question": "<question_text>", "options": ["<A>", "<B>", "<C>", "<D>"], "correct": <0-3>, "explanation": "<why>"}}]}}"""

def make_resource_prompt(topic_id, level='beginner'):
    t = TOPIC_GRAPH.get(topic_id)
    name = t.name if t else topic_id
    real_res = RESOURCE_DB.get(topic_id, [])
    hint = f"Known resources: {[r.title for r in real_res[:3]]}" if real_res else ""
    return f"""[RESOURCE] Recommend learning resources for: {name} (id: {topic_id})
Student level: {level}
{hint}
Respond with JSON: {{"resources": [{{"title": "<n>", "url": "<url>", "type": "<course|article|video>", "difficulty": "<beginner|intermediate|advanced>", "reason": "<why_this_resource>"}}]}}
Recommend 3-5 high-quality, real resources."""

random.seed(42)
all_prompts = []

# Action prompts (40%)
for seed in range(240):
    p = PROFILES[seed % len(PROFILES)]
    e = EduPathEnv(seed=seed)
    s = student_manager.create(name=f'mt_{seed}')
    student_manager.update_from_onboarding(s.id, {'target_field':p['field'],'learning_goal':p['goal'],'weekly_hours':p['hours']})
    o = e.reset(student_id=s.id, seed=seed)
    for _ in range(random.randint(0, 6)):
        avail = o.available_topics
        if not avail: break
        at = random.choice([ActionType.RECOMMEND_TOPIC, ActionType.ASSIGN_QUIZ, ActionType.RECOMMEND_RESOURCE])
        try:
            result = e.step(Action(type=at, topic_id=random.choice(avail) if avail else None))
            if result.done: break
            o = result.observation
        except: break
    all_prompts.append(make_action_prompt(o.model_dump()))

# Roadmap prompts (20%)
for i in range(120):
    p = PROFILES[i % len(PROFILES)]
    all_prompts.append(make_roadmap_prompt(p['field'], p['goal']))

# Quiz prompts (20%)
topic_list = list(ALL_TOPICS)
for i in range(120):
    all_prompts.append(make_quiz_prompt(topic_list[i % len(topic_list)]))

# Resource prompts (20%)
levels = ['beginner', 'intermediate', 'advanced']
for i in range(120):
    all_prompts.append(make_resource_prompt(topic_list[i % len(topic_list)], levels[i % 3]))

random.shuffle(all_prompts)
train_size   = min(3500, int(len(all_prompts) * 0.875))
train_prompts = all_prompts[:train_size]
eval_prompts  = all_prompts[train_size:]
assert len(train_prompts) >= 100, f'Too few train prompts: {len(train_prompts)}'
assert len(eval_prompts)  >= 10,  f'Too few eval prompts: {len(eval_prompts)}'

def wrap_chat(prompt_text):
    if len(prompt_text) > 1500:
        prompt_text = prompt_text[:1500] + '...[truncated]'
    return [
        {'role': 'system',  'content': 'You are an AI tutoring assistant. Always respond with valid JSON only, no explanation or markdown.'},
        {'role': 'user',    'content': prompt_text},
    ]

train_chat    = [wrap_chat(p) for p in train_prompts]
train_dataset = Dataset.from_dict({'prompt': train_chat})

task_dist = defaultdict(int)
for p in train_prompts: task_dist[detect_task(p)] += 1
print(f'Total prompts: {len(all_prompts)} | Train: {len(train_prompts)} | Eval: {len(eval_prompts)}')
print(f'Task distribution: {dict(task_dist)}')
print('Dataset ✅')


Total prompts: 600 | Train: 525 | Eval: 75
Task distribution: {'resource': 107, 'roadmap': 102, 'quiz': 104, 'action': 212}
Dataset ✅


In [4]:
# ==================== CELL 5: Model Loading ====================
MODEL_ID     = 'Qwen/Qwen2.5-3B-Instruct'
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = tokenizer = None

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        'unsloth/Qwen2.5-3B-Instruct-bnb-4bit', max_seq_length=1024, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=32, lora_dropout=0.05, bias='none', use_gradient_checkpointing='unsloth')
    print('Unsloth 3B 4-bit ✅')
except ImportError as e:
    print(f'Unsloth N/A ({e}), using HF')
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=compute_dtype)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map='auto', attn_implementation='eager')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '<|pad|>'})
        model.resize_token_embeddings(len(tokenizer))
    tokenizer.padding_side = 'left'
    from peft import get_peft_model, LoraConfig, TaskType
    lora = LoraConfig(r=16, lora_alpha=32,
        target_modules=['q_proj','k_proj','v_proj','o_proj'],
        lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM)
    model = get_peft_model(model, lora)
    model.config.use_cache = True
    if hasattr(model, 'base_model'):
        model.base_model.config.use_cache = True
    from transformers import GenerationConfig
    model.generation_config = GenerationConfig(
        temperature=0.8, top_p=0.95, top_k=50, do_sample=True,
        max_new_tokens=400,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id or tokenizer.pad_token_id,
    )
    print('HF 3B 4-bit ✅')

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: {total/1e6:.0f}M total, {trainable/1e6:.1f}M trainable')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f}GB')


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


Unsloth N/A (No module named 'unsloth'), using HF


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

HF 3B 4-bit ✅
Params: 1706M total, 7.4M trainable
VRAM used: 2.1GB


In [ ]:
# ==================== CELL 6: Baseline Evaluation ====================
import time

def evaluate_model(model, tokenizer, prompts, label='Model', n=40):
    model.eval()
    rewards_all = []; task_rewards = defaultdict(list)
    actions_count = {}; valid_json = 0
    device = next(model.parameters()).device

    print(f"\n{'='*60}")
    print(f"  Evaluating: {label}  ({n} prompts)")
    print(f"{'='*60}")
    print(f"  {'Step':>5}  {'Task':<10}  {'Reward':>8}  {'JSON':>6}  {'RunAvg':>8}  {'Time':>6}")
    print(f"  {'-'*55}")

    t_eval_start = time.time()

    try:
        for idx, p in enumerate(prompts[:n]):
            t_step = time.time()

            chat     = wrap_chat(p) if isinstance(p, str) else p
            inp_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
            inp      = tokenizer(inp_text, return_tensors='pt', truncation=True, max_length=512).to(device)

            with torch.no_grad():
                out = model.generate(
                    **inp, max_new_tokens=400, temperature=0.7, do_sample=True,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)

            text = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
            r    = multitask_reward([text], prompts=[p])[0]
            rewards_all.append(r)

            task = detect_task(p)
            task_rewards[task].append(r)

            got_json = False
            m = re.search(r'\{.*\}', text, re.DOTALL)
            if m:
                valid_json += 1; got_json = True
                try:
                    d = json.loads(m.group())
                    if 'type' in d:
                        actions_count[d['type']] = actions_count.get(d['type'], 0) + 1
                except: pass

            elapsed   = time.time() - t_step
            run_avg   = float(np.mean(rewards_all))
            json_mark = '✅' if got_json else '❌'

            print(f"  {idx+1:>5}/{n}  {task:<10}  {r:>+.4f}  {json_mark}     {run_avg:>+.4f}  {elapsed:>4.1f}s")

            # Print task breakdown every 10 steps
            if (idx + 1) % 10 == 0:
                print(f"  --- checkpoint {idx+1}/{n} ---")
                for t, rv in task_rewards.items():
                    print(f"      {t:<12}: mean={float(np.mean(rv)):+.3f}  n={len(rv)}")
                vram = torch.cuda.memory_allocated()/1e9
                print(f"      VRAM: {vram:.1f}GB  |  JSON rate so far: {valid_json/(idx+1):.0%}")
                print()

    finally:
        model.train()

    total_time = time.time() - t_eval_start
    per_task   = {k: float(np.mean(v)) for k, v in task_rewards.items()}

    print(f"\n{'='*60}")
    print(f"  {label} — Final Results")
    print(f"{'='*60}")
    print(f"  Mean reward  : {float(np.mean(rewards_all)):+.4f}")
    print(f"  Std dev      : {float(np.std(rewards_all)):.4f}")
    print(f"  Positive rate: {float(np.mean([r>0 for r in rewards_all])):.0%}")
    print(f"  Valid JSON   : {valid_json}/{n}  ({valid_json/max(n,1):.0%})")
    print(f"  Total time   : {total_time:.1f}s")
    print(f"  Per-task breakdown:")
    for t, avg in per_task.items():
        count = len(task_rewards[t])
        print(f"    {t:<12}: {avg:+.4f}  (n={count})")
    if actions_count:
        print(f"  Action distribution: {actions_count}")
    print(f"{'='*60}\n")

    return {
        'label': label, 'mean': float(np.mean(rewards_all)),
        'std': float(np.std(rewards_all)),
        'pos_rate': float(np.mean([r > 0 for r in rewards_all])),
        'json_rate': valid_json / max(n, 1),
        'actions': actions_count, 'per_task': per_task, 'rewards': rewards_all,
    }

_stats['actions'].clear(); _stats['total'] = 0
baseline = evaluate_model(model, tokenizer, eval_prompts, 'Baseline', min(40, len(eval_prompts)))
print('Baseline ✅')

Baseline eval (this may take a few minutes)...


In [ ]:
# ==================== CELL 7: GRPO Training (200 steps) ====================
import trl; from trl import GRPOTrainer, GRPOConfig
from datetime import datetime
import transformers as _hf_transformers
import time

# Patch GRPOTrainer._get_train_sampler — TRL 0.15.0 / Transformers signature mismatch
from torch.utils.data import RandomSampler
_original_sampler = GRPOTrainer._get_train_sampler
def _patched_get_train_sampler(self, train_dataset=None):
    try:
        return _original_sampler(self)
    except TypeError:
        return RandomSampler(self.train_dataset if train_dataset is None else train_dataset)
GRPOTrainer._get_train_sampler = _patched_get_train_sampler

_stats['actions'].clear(); _stats['total'] = 0; _stats['parse_fail'] = 0
_sig  = set(inspect.signature(GRPOConfig.__init__).parameters.keys())
_bf16 = torch.cuda.is_bf16_supported()

args = dict(
    output_dir                  = SAVE_DIR,
    max_steps                   = 200,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    num_generations             = 4,
    learning_rate               = 5e-5,
    lr_scheduler_type           = 'cosine',
    warmup_steps                = 20,
    weight_decay                = 0.01,
    max_grad_norm               = 0.3,
    logging_steps               = 1,
    save_steps                  = 50,
    save_total_limit            = 2,
    gradient_checkpointing      = False,
    report_to                   = 'none',
    seed                        = 42,
    bf16                        = _bf16,
    fp16                        = not _bf16,
)
if 'use_vllm'         in _sig: args['use_vllm']         = False
if 'beta'             in _sig: args['beta']             = 0.04
if 'max_prompt_length' in _sig: args['max_prompt_length'] = 384
if 'temperature'      in _sig: args['temperature']      = 0.8
if 'top_p'            in _sig: args['top_p']            = 0.95
for c in ['max_completion_length', 'max_new_tokens', 'generation_max_new_tokens']:
    if c in _sig: args[c] = 400; break
if not any(c in args for c in ['max_completion_length', 'max_new_tokens', 'generation_max_new_tokens']):
    args['max_completion_length'] = 400

config = GRPOConfig(**args)

# ── Detailed step-by-step callback ──────────────────────────────────────────
class VerboseCallback(_hf_transformers.TrainerCallback):

    def __init__(self):
        self.step_times = []
        self.rewards    = []
        self.losses     = []
        self.t_step     = None
        self.t_start    = None

    def on_train_begin(self, args, state, control, **kwargs):
        print(f"\n{'='*65}")
        print(f"  GRPO Training started — {args.max_steps} steps")
        print(f"  batch={args.per_device_train_batch_size}  "
              f"grad_accum={args.gradient_accumulation_steps}  "
              f"lr={args.learning_rate}  beta=0.04")
        print(f"  VRAM at start: {torch.cuda.memory_allocated()/1e9:.1f} GB")
        print(f"{'='*65}")
        print(f"  {'Step':>6}  {'Loss':>8}  {'Reward':>8}  {'Δbaseline':>10}  "
              f"{'StepTime':>9}  {'ETA':>8}  {'VRAM':>6}")
        print(f"  {'-'*63}")
        self.t_start = time.time()
        self.t_step  = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        step   = state.global_step
        loss   = logs.get('loss', None)
        reward = logs.get('rewards/mean', logs.get('reward', logs.get('train/reward', None)))

        now       = time.time()
        step_time = now - self.t_step if self.t_step else 0
        self.t_step = now
        self.step_times.append(step_time)

        if loss   is not None: self.losses.append(float(loss))
        if reward is not None: self.rewards.append(float(reward))

        avg_step_time = float(np.mean(self.step_times[-20:])) if self.step_times else 0
        eta_sec       = avg_step_time * (args.max_steps - step)
        eta_str       = f"{eta_sec/60:.1f}m" if eta_sec > 60 else f"{eta_sec:.0f}s"

        vram      = torch.cuda.memory_allocated()/1e9
        delta     = float(reward) - baseline['mean'] if reward is not None else 0.0
        loss_str  = f"{float(loss):.4f}"   if loss   is not None else "  n/a "
        rew_str   = f"{float(reward):+.4f}" if reward is not None else "  n/a "
        delta_str = f"{delta:+.4f}"         if reward is not None else "  n/a "

        print(f"  {step:>6}/{args.max_steps}  {loss_str:>8}  {rew_str:>8}  "
              f"{delta_str:>10}  {step_time:>7.1f}s  {eta_str:>8}  {vram:>4.1f}GB")

        # Mini summary every 25 steps
        if step % 25 == 0 and step > 0:
            avg_r   = float(np.mean(self.rewards[-25:])) if self.rewards else 0
            avg_l   = float(np.mean(self.losses[-25:]))  if self.losses  else 0
            elapsed = (now - self.t_start) / 60
            print(f"\n  ── Step {step} summary ──────────────────────────────")
            print(f"     Avg reward (last 25) : {avg_r:+.4f}")
            print(f"     Avg loss   (last 25) : {avg_l:.4f}")
            print(f"     Elapsed              : {elapsed:.1f} min")
            print(f"     Parse fails total    : {_stats['parse_fail']}")
            print(f"     Action distribution  : {dict(_stats['actions'])}")
            print(f"  {'─'*53}\n")

        # Save status.json for external monitoring
        with open(f'{WORK}/status.json', 'w') as f:
            json.dump({
                'status' : 'training',
                'step'   : step,
                'total'  : args.max_steps,
                'reward' : float(reward) if reward is not None else None,
                'loss'   : float(loss)   if loss   is not None else None,
                'eta'    : eta_str,
                'vram_gb': round(vram, 2),
            }, f)

    def on_save(self, args, state, control, **kwargs):
        print(f"  💾 Checkpoint saved at step {state.global_step} → {args.output_dir}")

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = (time.time() - self.t_start) / 60
        avg_r   = float(np.mean(self.rewards)) if self.rewards else 0
        print(f"\n{'='*65}")
        print(f"  Training complete!")
        print(f"  Total time   : {elapsed:.1f} min")
        print(f"  Steps done   : {state.global_step}")
        print(f"  Avg reward   : {avg_r:+.4f}  (baseline was {baseline['mean']:+.4f})")
        print(f"  Improvement  : {avg_r - baseline['mean']:+.4f}")
        print(f"  Parse fails  : {_stats['parse_fail']}")
        print(f"  VRAM peak    : {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
        print(f"{'='*65}\n")

# ── Build & run trainer ──────────────────────────────────────────────────────
if not hasattr(model, 'warnings_issued'):
    model.warnings_issued = {}

kw = dict(model=model, args=config, train_dataset=train_dataset, reward_funcs=[multitask_reward])
if trl.__version__ >= '0.15.0': kw['processing_class'] = tokenizer
else:                             kw['tokenizer']        = tokenizer

trainer = GRPOTrainer(**kw)
trainer.add_callback(VerboseCallback())

mins = 0.0
t0   = datetime.now()
try:
    result = trainer.train()
    mins   = (datetime.now() - t0).total_seconds() / 60
except Exception as e:
    import traceback
    with open(f'{WORK}/status.json', 'w') as f:
        json.dump({'status':'error','error':str(e),'traceback':traceback.format_exc()}, f)
    print(f'Training crashed: {e}'); raise

print(f'Done in {mins:.1f} min ✅')

In [ ]:
# ==================== CELL 8: Post-Training Evaluation ====================
_stats['actions'].clear(); _stats['total'] = 0
trained = evaluate_model(model, tokenizer, eval_prompts, 'Trained', 40)
imp = trained['mean'] - baseline['mean']
print(f"  Mean: {trained['mean']:+.3f} | Pos: {trained['pos_rate']:.0%} | JSON: {trained['json_rate']:.0%}")
print(f"  Per-task: {trained['per_task']}")
print(f"  Improvement: {imp:+.3f} ({'BETTER' if imp > 0 else 'WORSE'})")


In [ ]:
# ==================== CELL 9: Plots & Results ====================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

log = trainer.state.log_history if hasattr(trainer, 'state') else []
st  = [e['step'] for e in log if 'loss' in e]
lo  = [e['loss'] for e in log if 'loss' in e]
rs  = [e['step'] for e in log if 'reward' in e or 'rewards/mean' in e]
rw  = [e.get('reward', e.get('rewards/mean', 0)) for e in log if 'reward' in e or 'rewards/mean' in e]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(st, lo, 'royalblue', lw=1, alpha=0.4)
if len(lo) > 5:
    axes[0, 0].plot(st, uniform_filter1d(lo, min(5, len(lo))), 'royalblue', lw=2.5, label='Smooth')
axes[0, 0].set(xlabel='Step', ylabel='Loss', title='Training Loss'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

if rs:
    axes[0, 1].plot(rs, rw, 'forestgreen', lw=1, alpha=0.4)
    if len(rw) > 5:
        axes[0, 1].plot(rs, uniform_filter1d(rw, min(5, len(rw))), 'forestgreen', lw=2.5, label='Smooth')
    axes[0, 1].axhline(baseline['mean'], color='red', ls='--', lw=1.5, label=f"Baseline ({baseline['mean']:+.2f})")
    axes[0, 1].legend()
axes[0, 1].set(xlabel='Step', ylabel='Reward', title='Multi-Task Reward'); axes[0, 1].grid(True, alpha=0.3)

lbl = ['Mean\nReward', 'Positive\nRate', 'Valid\nJSON']
bv  = [baseline['mean'], baseline['pos_rate'], baseline['json_rate']]
av  = [trained['mean'],  trained['pos_rate'],  trained['json_rate']]
x, w = np.arange(3), 0.35
b1 = axes[1, 0].bar(x - w/2, bv, w, label='Before', color='#e74c3c', alpha=0.85)
b2 = axes[1, 0].bar(x + w/2, av, w, label='After',  color='#2ecc71', alpha=0.85)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}', ha='center', fontsize=9)
axes[1, 0].set_xticks(x); axes[1, 0].set_xticklabels(lbl)
axes[1, 0].set(ylabel='Score', title='Before vs After GRPO'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3, axis='y')

tasks = list(set(list(baseline.get('per_task',{}).keys()) + list(trained.get('per_task',{}).keys())))
if tasks:
    bpt = [baseline.get('per_task',{}).get(t, 0) for t in tasks]
    apt = [trained.get('per_task',{}).get(t, 0)  for t in tasks]
    x2, w2 = np.arange(len(tasks)), 0.35
    axes[1, 1].bar(x2 - w2/2, bpt, w2, label='Before', color='#e74c3c', alpha=0.85)
    axes[1, 1].bar(x2 + w2/2, apt, w2, label='After',  color='#2ecc71', alpha=0.85)
    axes[1, 1].set_xticks(x2); axes[1, 1].set_xticklabels(tasks, fontsize=9)
    axes[1, 1].set(ylabel='Reward', title='Per-Task Improvement'); axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle('EduPath AI — Multi-Task GRPO (3B)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORK}/grpo_training_results.png', dpi=150, bbox_inches='tight')
plt.show()

results = {
    'model': MODEL_ID, 'steps': config.max_steps, 'time_min': round(mins, 1),
    'baseline': {'mean': baseline['mean'], 'pos_rate': baseline['pos_rate'],
                 'json_rate': baseline['json_rate'], 'per_task': baseline.get('per_task', {})},
    'trained' : {'mean': trained['mean'],  'pos_rate': trained['pos_rate'],
                 'json_rate': trained['json_rate'],  'per_task': trained.get('per_task', {})},
    'improvement': round(imp, 4),
}
with open(f'{WORK}/grpo_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved plot + JSON ✅  →  {WORK}/grpo_training_results.png')


In [ ]:
# ==================== CELL 10: Save Model & (Optional) HF Upload ====================
trainer.save_model(FINAL); tokenizer.save_pretrained(FINAL)
print(f'Model saved to {FINAL} ✅')

print(f'\n{"="*60}')
print(f'  Model:  {MODEL_ID} (3B)')
print(f'  Steps:  {config.max_steps} | Time: {mins:.1f}min')
print(f'  Before: reward={baseline["mean"]:+.3f} pos={baseline["pos_rate"]:.0%} json={baseline["json_rate"]:.0%}')
print(f'  After:  reward={trained["mean"]:+.3f} pos={trained["pos_rate"]:.0%} json={trained["json_rate"]:.0%}')
print(f'  Delta:  {imp:+.3f}')
for t in ['action', 'roadmap', 'quiz', 'resource']:
    b = baseline.get('per_task', {}).get(t, 0)
    a = trained.get('per_task', {}).get(t, 0)
    print(f'    {t:10s}: {b:+.3f} → {a:+.3f} ({("+" if a > b else "")}{a-b:.3f})')
print(f'{"="*60}')

# ---- Optional HuggingFace upload ----
# Set HF_TOKEN in Colab:  from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    from peft import AutoPeftModelForCausalLM
    from huggingface_hub import HfApi, login
    import gc
    login(token=HF_TOKEN)
    print('Merging LoRA adapters...')
    gc.collect(); torch.cuda.empty_cache()
    merged = AutoPeftModelForCausalLM.from_pretrained(FINAL, dtype=torch.float16, device_map='auto')
    merged = merged.merge_and_unload()
    merged.save_pretrained(f'{WORK}/merged_v3')
    tokenizer.save_pretrained(f'{WORK}/merged_v3')
    api = HfApi()
    api.create_repo('degree-checker-01/edupath-grpo-tutor', repo_type='model', exist_ok=True)
    api.upload_folder(folder_path=f'{WORK}/merged_v3', repo_id='degree-checker-01/edupath-grpo-tutor', repo_type='model')
    api.upload_file(path_or_fileobj=f'{WORK}/grpo_training_results.png',
                    path_in_repo='grpo_training_results.png',
                    repo_id='degree-checker-01/edupath-grpo-tutor', repo_type='model')
    api.upload_file(path_or_fileobj=f'{WORK}/grpo_results.json',
                    path_in_repo='grpo_results.json',
                    repo_id='degree-checker-01/edupath-grpo-tutor', repo_type='model')
    print('✅ Model + evidence uploaded to HuggingFace')
else:
    print('⚠️  HF_TOKEN not set — skipping upload.')
    print('   To upload, run: os.environ["HF_TOKEN"] = "hf_..."  then re-run this cell.')
